# Workshop Template — Apply PEFT to Your Own Model and Data

This notebook is a **fill-in-the-blanks template**. It runs end-to-end with sensible
defaults so you can verify the setup works, then swap in your own model, dataset and
adaptation method.

**Workflow:**

```
1. Config      ← set your choices once at the top
2. Model       ← load any ViT-style backbone (HF or custom)
3. Dataset     ← load any image classification dataset
4. PEFT method ← pick one from src/methods and configure it
5. Train       ← standard training loop from src/training
6. Evaluate    ← accuracy curve + trainable-param summary
```

Every cell marked `# ── USER SECTION ──` is meant to be edited.
Cells without that marker can be left as-is.

In [ ]:
import sys, os

if "google.colab" in sys.modules or "COLAB_GPU" in os.environ:
    if not os.path.exists("haicon_peft_co"):
        os.system("git clone https://github.com/trofimova/haicon_peft_co.git")
    os.chdir("haicon_peft_co")
    os.system("pip install -q transformers peft torchvision")

# make sure src/ is importable
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())


## 0. Setup

In [ ]:
# !pip install -q torch torchvision transformers peft matplotlib pandas tqdm
import sys, copy
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
from transformers import ViTModel

ROOT = Path.cwd()
if (ROOT / ".." / "src").exists():
    sys.path.insert(0, str((ROOT / "..").resolve()))
elif (ROOT / "src").exists():
    sys.path.insert(0, str(ROOT.resolve()))

from src.methods.linear_probe import LinearProbeModel
from src.methods.adapters import AdapterHeadClassifier
from src.methods.prompt_tuning import PromptTunedClassifier
from src.methods.lora import LoRAClassifier
from src.methods.bitfit import BitFitClassifier
from src.methods.partial_ft import PartialFineTuneClassifier
from src.training import count_trainable_parameters, freeze_module, train_model, evaluate

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

---
## 1. Configuration

**Edit this cell.** Everything downstream reads from these variables.

In [ ]:
# ── USER SECTION ──────────────────────────────────────────────────────────────

# --- Model ---
MODEL_NAME = "WinKawaks/vit-small-patch16-224"   # any HF ViT-style model
IMG_SIZE   = 224                                  # must match model

# --- Dataset ---
# Default: CIFAR-10. Increase MAX_*_SAMPLES for better accuracy (but longer runs).
N_CLASSES         = 10
BATCH_SIZE        = 16
MAX_TRAIN_SAMPLES = 500   # use None to train on the full dataset
MAX_VAL_SAMPLES   = 100

# --- Adaptation method ---
# Choose one of:
#   "linear_probe" | "bitfit" | "visual_prompt" | "lora" | "adapter" | "partial_ft"
METHOD = "lora"

# --- Training ---
EPOCHS            = 5
COMPARISON_EPOCHS = 3     # epochs used in Section 7 multi-method comparison
LR                = 3e-3
# ── END USER SECTION ──────────────────────────────────────────────────────────

print("Method :", METHOD)
print("Model  :", MODEL_NAME)
print(f"Subset : {MAX_TRAIN_SAMPLES} train / {MAX_VAL_SAMPLES} val")


---
## 2. Load the Backbone

The default loads a HuggingFace ViT and wraps it so it returns the CLS token.

**To swap in your own model:**
- Define any `nn.Module` that takes `(B, 3, H, W)` images and returns `(B, D)` features.
- Set `backbone.feature_dim = D` so the PEFT wrappers know the feature size.

In [ ]:
# ── USER SECTION (optional — swap in your own backbone) ──────────────────────

class HFViTBackbone(nn.Module):
    """Thin wrapper around a HuggingFace ViTModel that returns the CLS token."""
    def __init__(self, model_name=MODEL_NAME):
        super().__init__()
        self.vit = ViTModel.from_pretrained(model_name)
        self.feature_dim = self.vit.config.hidden_size

    def forward(self, x):
        return self.vit(pixel_values=x).last_hidden_state[:, 0]

# --- Replace the line below to use a different backbone ---
backbone = HFViTBackbone().to(device)
# ── END USER SECTION ──────────────────────────────────────────────────────────

freeze_module(backbone)
D     = backbone.feature_dim
total = sum(p.numel() for p in backbone.parameters())
print(f"Backbone     : {MODEL_NAME}")
print(f"Feature dim  : {D}")
print(f"Total params : {total:,}")

---
## 3. Load the Dataset

**Default:** CIFAR-10, resized to 224×224.  
To use your own data, replace the dataset cell below with any `torch.utils.data.Dataset` that returns `(image_tensor, label)`.


In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import Subset
import random

tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_full = datasets.CIFAR10("/tmp/cifar10", train=True,  download=True, transform=tfm)
val_full   = datasets.CIFAR10("/tmp/cifar10", train=False, download=True, transform=tfm)

def subsample(ds, n):
    if n is None or n >= len(ds):
        return ds
    idx = random.sample(range(len(ds)), n)
    return Subset(ds, idx)

train_ds = subsample(train_full, MAX_TRAIN_SAMPLES)
val_ds   = subsample(val_full,   MAX_VAL_SAMPLES)

# ── Bring your own dataset (uncomment and replace) ────────────────────────────
# from torchvision.datasets import ImageFolder
# train_full = ImageFolder("data/train", transform=tfm)
# val_full   = ImageFolder("data/val",   transform=tfm)
# N_CLASSES  = len(train_full.classes)
# train_ds, val_ds = subsample(train_full, MAX_TRAIN_SAMPLES), subsample(val_full, MAX_VAL_SAMPLES)
# ─────────────────────────────────────────────────────────────────────────────

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train samples : {len(train_ds):,}")
print(f"Val   samples : {len(val_ds):,}")
print(f"Classes       : {N_CLASSES}")


---
## 4. Build the Adapted Model

The factory below reads `METHOD` from the config cell and instantiates the right wrapper
from `src/methods/`. You can also call a wrapper directly if you want fine-grained control.

In [ ]:
from src.methods.configs import CONFIGS

def build_model(method: str, backbone: nn.Module, D: int, K: int) -> nn.Module:
    """Instantiate the chosen PEFT wrapper using the default config for that method.

    To override a hyperparameter, edit src/methods/configs.py before running.
    """
    cfg = CONFIGS[method]
    bb  = copy.deepcopy(backbone)

    if method == "linear_probe":
        return LinearProbeModel(bb, D, K)

    if method == "bitfit":
        return BitFitClassifier(bb, D, K)

    if method == "visual_prompt":
        return PromptTunedClassifier(bb, D, K,
                                     num_prompt_tokens=cfg.num_prompt_tokens)

    if method == "lora":
        return LoRAClassifier(bb, D, K,
                              target_modules=cfg.target_modules, rank=cfg.rank)

    if method == "adapter":
        return AdapterHeadClassifier(bb, D, K, bottleneck_dim=cfg.bottleneck_dim)

    if method == "partial_ft":
        blocks = list(bb.vit.encoder.layer)   # adjust for custom backbones
        return PartialFineTuneClassifier(bb, D, K,
                                         modules_to_unfreeze=blocks[-cfg.n_blocks:])

    raise ValueError(f"Unknown method '{method}'. "
                     "Choose from: linear_probe, bitfit, visual_prompt, "
                     "lora, adapter, partial_ft")


model = build_model(METHOD, backbone, D, N_CLASSES).to(device)

total_p = sum(p.numel() for p in model.parameters())
train_p = count_trainable_parameters(model)
cfg = CONFIGS[METHOD]
print(f"Method            : {METHOD}")
print(f"Config            : {cfg}")
print(f"Trainable params  : {train_p:,}")
print(f"Total params      : {total_p:,}")
print(f"% of total        : {100*train_p/total_p:.3f}%")
print()
print("Trainable tensors:")
for name, p in model.named_parameters():
    if p.requires_grad:
        print(f"  {name:55s} {list(p.shape)}")


---
## 5. Train

In [ ]:
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR, weight_decay=1e-2
)

history = train_model(model, train_loader, val_loader,
                      optimizer, epochs=EPOCHS, device=device)

print(f"\nFinal val accuracy : {history.val_acc[-1]:.3f}")

---
## 6. Evaluate and Visualise

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

axes[0].plot(history.train_loss, label="train loss")
axes[0].plot(history.val_loss,   label="val loss")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].set_title(f"{METHOD} — loss")
axes[0].legend()

axes[1].plot(history.val_acc, color="steelblue")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Val accuracy")
axes[1].set_title(f"{METHOD} — validation accuracy")
axes[1].set_ylim(0, 1)

plt.suptitle(f"Training results  |  method={METHOD}  |  trainable={train_p:,} params",
             fontsize=12)
plt.tight_layout()
plt.show()

---
## 7. (Optional) Compare Multiple Methods

Run this section to train all six methods on the same data and compare their
final validation accuracy against trainable parameter count.

In [ ]:
import time, gc

ALL_METHODS = ["linear_probe", "bitfit", "visual_prompt", "lora", "adapter", "partial_ft"]

def _dev_type(dev):
    return torch.device(dev).type if isinstance(dev, str) else dev.type

def reset_mem(dev):
    t = _dev_type(dev)
    if t == "cuda":
        torch.cuda.reset_peak_memory_stats(dev)
    elif t == "mps":
        torch.mps.empty_cache()

def peak_mem_mb(dev):
    t = _dev_type(dev)
    if t == "cuda":
        return torch.cuda.max_memory_allocated(dev) / 1024**2
    elif t == "mps":
        return torch.mps.current_allocated_memory() / 1024**2
    return float("nan")   # CPU: not tracked

def infer_ms_per_img(model, loader, dev, n_batches=5):
    model.eval()
    t = _dev_type(dev)
    times = []
    with torch.no_grad():
        for i, (x, _) in enumerate(loader):
            if i >= n_batches:
                break
            x = x.to(dev)
            if t == "cuda":
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            model(x)
            if t == "cuda":
                torch.cuda.synchronize()
            times.append((time.perf_counter() - t0) / x.size(0) * 1000)
    return sum(times) / len(times)

results = []
for m_name in ALL_METHODS:
    print(f"\n[{m_name}]", flush=True)
    gc.collect()
    reset_mem(device)

    m = build_model(m_name, backbone, D, N_CLASSES).to(device)
    opt = torch.optim.AdamW(
        [p for p in m.parameters() if p.requires_grad], lr=LR, weight_decay=1e-2
    )

    t_start = time.perf_counter()
    hist = train_model(m, train_loader, val_loader, opt, epochs=COMPARISON_EPOCHS, device=device)
    train_time_s = time.perf_counter() - t_start

    infer_ms = infer_ms_per_img(m, val_loader, device)
    mem_mb   = peak_mem_mb(device)
    tr       = count_trainable_parameters(m)
    total_p  = sum(p.numel() for p in m.parameters())

    results.append({
        "method":        m_name,
        "trainable":     tr,
        "% params":      f"{100 * tr / total_p:.3f}",
        "val_acc":       hist.val_acc[-1],
        "train_time_s":  round(train_time_s, 1),
        "infer_ms/img":  round(infer_ms, 3),
        "peak_mem_mb":   round(mem_mb, 1) if not __import__("math").isnan(mem_mb) else "n/a",
        "history":       hist,
    })
    print(f"  val_acc={hist.val_acc[-1]:.3f}  params={tr:,}  "
          f"train={train_time_s:.0f}s  infer={infer_ms:.2f}ms/img  mem={mem_mb:.0f}MB")

df_cmp = pd.DataFrame([{k: v for k, v in r.items() if k != "history"}
                        for r in results]).set_index("method")
print()
print(df_cmp.to_string())


In [ ]:
fig = plt.figure(figsize=(15, 8))
gs  = fig.add_gridspec(2, 3, hspace=0.45, wspace=0.38)

# ── Top-left: validation accuracy curves ──────────────────────────────────────
ax0 = fig.add_subplot(gs[0, 0])
for r in results:
    ax0.plot(r["history"].val_acc, label=r["method"])
ax0.set_xlabel("Epoch"); ax0.set_ylabel("Val accuracy")
ax0.set_title("Validation accuracy")
ax0.legend(fontsize=7); ax0.set_ylim(0, 1)

# ── Top-centre: accuracy vs trainable params ──────────────────────────────────
ax1 = fig.add_subplot(gs[0, 1])
ax1.scatter(df_cmp["trainable"], df_cmp["val_acc"], s=80, zorder=3)
for method, row in df_cmp.iterrows():
    ax1.annotate(method, (row["trainable"], row["val_acc"]),
                 textcoords="offset points", xytext=(5, 3), fontsize=7)
ax1.set_xscale("log"); ax1.set_xlabel("Trainable params (log)")
ax1.set_ylabel("Final val accuracy")
ax1.set_title("Accuracy vs efficiency"); ax1.grid(True, alpha=0.3)

# ── Top-right: training time (s) ──────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 2])
times = [r["train_time_s"] for r in results]
methods = [r["method"] for r in results]
ax2.barh(methods, times, color="#4c72b0")
ax2.set_xlabel("Total training time (s)")
ax2.set_title("Training time")
for bar, v in zip(ax2.patches, times):
    ax2.text(v + max(times)*0.01, bar.get_y() + bar.get_height()/2,
             f"{v:.0f}s", va="center", fontsize=7)

# ── Bottom-left: inference latency ────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
infer = [r["infer_ms/img"] for r in results]
ax3.barh(methods, infer, color="#dd8452")
ax3.set_xlabel("Inference latency (ms / image)")
ax3.set_title("Inference speed")
for bar, v in zip(ax3.patches, infer):
    ax3.text(v + max(infer)*0.01, bar.get_y() + bar.get_height()/2,
             f"{v:.2f}", va="center", fontsize=7)

# ── Bottom-centre: peak memory ────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
mem_vals = [r["peak_mem_mb"] if r["peak_mem_mb"] != "n/a" else 0 for r in results]
bars = ax4.barh(methods, mem_vals, color="#55a868")
ax4.set_xlabel("Peak memory (MB)")
ax4.set_title("Memory usage")
has_mem = any(v > 0 for v in mem_vals)
if not has_mem:
    ax4.text(0.5, 0.5, "not available on CPU", ha="center", va="center",
             transform=ax4.transAxes, fontsize=9, color="#888")
for bar, v in zip(bars, mem_vals):
    if v > 0:
        ax4.text(v + max(mem_vals)*0.01, bar.get_y() + bar.get_height()/2,
                 f"{v:.0f}", va="center", fontsize=7)

# ── Bottom-right: trainable param % ───────────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 2])
pct = [float(r["% params"]) for r in results]
ax5.barh(methods, pct, color="#c44e52")
ax5.set_xlabel("Trainable parameters (%)")
ax5.set_title("Parameter overhead")
for bar, v in zip(ax5.patches, pct):
    ax5.text(v + max(pct)*0.01, bar.get_y() + bar.get_height()/2,
             f"{v:.3f}%", va="center", fontsize=7)

plt.suptitle("PEFT method comparison — accuracy & efficiency", fontsize=13, y=1.01)
plt.show()
